<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/01_download_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/01_download_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Download datasets

Downloads every raw dataset into `MyDrive/clear_vision/datasets/raw/<name>/`.

- **Safe to re-run.** A dataset is downloaded to the Colab disk first, copied to Drive,
  and only then marked with a `.done` file. Finished datasets are skipped. A folder
  without `.done` is a partial download and is replaced on the next run.
- **Before running:** add `ROBOFLOW_API_KEY` under Secrets (the key icon on the left)
  and turn on notebook access. For section 5, upload the Mendeley `.zip` files to
  `MyDrive/clear_vision/datasets/raw/mendeley/`.
- No GPU is needed. Run all cells, then check the summary table at the end.


In [8]:
# Settings: edit these lists, then run all cells.

# Where to download:
#   "drive": MyDrive/clear_vision/datasets/raw/ (kept between sessions)
#   "local": /content/raw/ on the Colab disk (fast, but gone when the runtime ends).
#            Use it for new sources that notebook 02 appends with APPEND = True,
#            in the same runtime.
RAW_TARGET = "drive"
# Only download these folder names; empty = all. Example: ["rf_sidewalk", "rf_barricade"]
ONLY: list[str] = []

# (workspace, project, version, folder name under datasets/raw/)
# If a version fails, the project's latest version is used instead.
# Removed: blindspot-fmcu0/sidewalk-obstacles (its download failed: no versions).
ROBOFLOW_DATASETS = [
    ("blindwalk", "bollard-v16ec", 1, "rf_bollard"),
    ("object-detection-3n5oe", "final-detection-9g3ji", 6, "rf_pole_curb_stairs"),
    ("kyonggi-university-r7unh", "protection_zone", 4, "rf_scooter_bollard"),
    ("tom-lai-8bp7n", "stairs-i2yia", 3, "rf_stairs"),
    ("georgia-institute-of-technology-bwqcc", "pothole-detection-mev9q", 6, "rf_pothole"),
    ("distracted-driving-yu8re", "pedestrian-crosswalk-c0irg", 1, "rf_crosswalk"),
    ("grape-disease-prediction", "general-cone-detection-ow4vb", 2, "rf_cone"),
    ("workspace-an3qw", "manhole-j9klz", 2, "rf_manhole"),
    ("scottsdale", "sidewalk-otzhb", 1, "rf_sidewalk"),
    ("myplace-65bsf", "barricade-detection-kaaty", 10, "rf_barricade"),
]

# Open Images V7: class names exactly as Open Images spells them.
OI_CLASSES = ["Stairs", "Traffic light", "Fire hydrant", "Bench", "Bicycle", "Person", "Car", "Dog"]
OI_MAX_PER_CLASS = 1500  # per class and per split (train, validation)

# COCO-2017 subset.
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "bus",
    "truck", "dog", "bench", "fire hydrant", "traffic light",
]
COCO_MAX_SAMPLES = {"train": 5000, "validation": 1000}  # set validation to 0 to skip it


## 1. Setup

In [2]:
# 1. Setup: Drive, packages, paths, API key, helpers
from google.colab import drive

drive.mount("/content/drive")

%pip install -q roboflow fiftyone

import json
import os
import shutil
import time
import zipfile
from pathlib import Path

from google.colab import userdata

DRIVE_ROOT = Path("/content/drive/MyDrive/clear_vision")
if RAW_TARGET not in ("drive", "local"):
    raise ValueError(f'RAW_TARGET must be "drive" or "local", got {RAW_TARGET!r}')
RAW = DRIVE_ROOT / "datasets" / "raw" if RAW_TARGET == "drive" else Path("/content/raw")
STAGING = Path("/content/staging")  # Colab disk: much faster than writing file by file to Drive
RAW.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)

DONE = ".done"  # written last, so a folder without it is a partial download
FAILURES: dict[str, str] = {}

try:
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    print("ROBOFLOW_API_KEY loaded")
except Exception as e:
    ROBOFLOW_API_KEY = None
    print(f"WARNING: ROBOFLOW_API_KEY not available ({type(e).__name__}).")
    print("         Add it under Secrets (key icon) with notebook access, then re-run.")
    print("         Roboflow datasets will be skipped.")


def wanted(name: str) -> bool:
    """True if `name` should be downloaded in this run (see ONLY)."""
    return not ONLY or name in ONLY


def is_done(dest: Path) -> bool:
    return (dest / DONE).exists()


def skip_if_done(dest: Path) -> bool:
    if is_done(dest):
        print(f"[{dest.name}] already downloaded, skipping")
        return True
    if dest.exists():
        print(f"[{dest.name}] partial download found, it will be replaced")
    return False


def fresh_staging(name: str) -> Path:
    local = STAGING / name
    shutil.rmtree(local, ignore_errors=True)
    return local


def publish(local: Path, dest: Path, info: dict) -> None:
    """Copy a finished download from the Colab disk to Drive, then mark it done."""
    t0 = time.time()
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(local, dest)
    (dest / DONE).write_text(json.dumps(info, indent=2))
    shutil.rmtree(local, ignore_errors=True)
    print(f"  copied to {dest} in {time.time() - t0:.0f}s")


def record_failure(name: str, error: Exception) -> None:
    FAILURES[name] = f"{type(error).__name__}: {error}"
    print(f"  FAILED: {FAILURES[name]}")


print(f"Raw datasets folder: {RAW}  (RAW_TARGET = {RAW_TARGET!r})")
if ONLY:
    print(f"Only downloading: {ONLY}")


Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
ROBOFLOW_API_KEY loaded
Raw datasets folder: /content/drive/MyDrive/clear_vision/datasets/raw


## 2. Roboflow datasets (YOLOv8)

In [3]:
# 2. Roboflow datasets (YOLOv8 format)
from roboflow import Roboflow


def version_number(version) -> int:
    return int(str(version.version).split("/")[-1])


def download_roboflow(rf, workspace: str, project_id: str, version: int, local: Path) -> int:
    """Download one Roboflow version; fall back to the latest one. Returns the version used."""
    project = rf.workspace(workspace).project(project_id)
    try:
        project.version(version).download("yolov8", location=str(local), overwrite=True)
        return version
    except Exception as e:
        print(f"  version {version} failed ({type(e).__name__}: {e}), trying the latest version")
    versions = project.versions()
    if not versions:
        raise RuntimeError(f"{workspace}/{project_id} has no versions")
    latest = max(version_number(v) for v in versions)
    shutil.rmtree(local, ignore_errors=True)
    project.version(latest).download("yolov8", location=str(local), overwrite=True)
    print(f"  USED LATEST VERSION {latest} instead of {version}")
    return latest


rf = Roboflow(api_key=ROBOFLOW_API_KEY) if ROBOFLOW_API_KEY else None

for workspace, project_id, version, name in ROBOFLOW_DATASETS:
    if not wanted(name):
        continue
    dest = RAW / name
    if skip_if_done(dest):
        continue
    print(f"\n[{name}] {workspace}/{project_id} v{version}")
    if rf is None:
        record_failure(name, RuntimeError("no ROBOFLOW_API_KEY"))
        continue
    local = fresh_staging(name)
    t0 = time.time()
    try:
        used = download_roboflow(rf, workspace, project_id, version, local)
        print(f"  downloaded v{used} in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "roboflow", "workspace": workspace, "project": project_id,
            "requested_version": version, "version": used, "format": "yolov8",
        })
    except Exception as e:
        record_failure(name, e)

print("\nRoboflow section finished")

# Compare each dataset's classes with its mapping file (configs/mappings/ on GitHub).
import urllib.request

import yaml

MAPPINGS_URL = "https://raw.githubusercontent.com/Zikirullo/clear-vision/develop/configs/mappings/{}.yaml"


def check_mapping(name: str) -> None:
    yaml_path = next((RAW / name / f for f in ("data.yaml", "dataset.yaml") if (RAW / name / f).exists()), None)
    if yaml_path is None:
        return
    names = (yaml.safe_load(yaml_path.read_text()) or {}).get("names") or []
    names = [str(names[k]) for k in sorted(names)] if isinstance(names, dict) else [str(n) for n in names]
    try:
        with urllib.request.urlopen(MAPPINGS_URL.format(name), timeout=10) as response:
            mapping = yaml.safe_load(response.read()) or {}
    except Exception as e:
        print(f"[{name}] no mapping file on GitHub ({e}). Classes: {names}")
        return
    cmap = {str(k): str(v) for k, v in (mapping.get("map") or {}).items()}
    unmapped = [n for n in names if n not in cmap]
    missing = [c for c in cmap if c not in names]
    if unmapped and mapping.get("unlisted") == "drop":
        print(f"[{name}] {len(unmapped)} unmapped classes, dropped by `unlisted: drop`: {unmapped}")
    elif unmapped:
        print(f"[{name}] UNMAPPED classes {unmapped}: the merge fails until they are in the mapping file")
    else:
        print(f"[{name}] all {len(names)} classes are mapped")
    if missing:
        print(f"[{name}] mapping lists classes this dataset does not have: {missing}")


print("\nClass check against configs/mappings/:")
for *_, name in ROBOFLOW_DATASETS:
    if wanted(name):
        check_mapping(name)



[rf_sidewalk_obstacles] blindspot-fmcu0/sidewalk-obstacles v1
loading Roboflow workspace...
loading Roboflow project...
  version 1 failed (RuntimeError: Version number 1 is not found.), trying the latest version
  FAILED: RuntimeError: blindspot-fmcu0/sidewalk-obstacles has no versions

[rf_bollard] blindwalk/bollard-v16ec v1
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/staging/rf_bollard in yolov8:: 100%|██████████| 139/139 [00:00<00:00, 5611.24it/s]


  downloaded v1 in 1s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/rf_bollard in 2s

[rf_pole_curb_stairs] object-detection-3n5oe/final-detection-9g3ji v6
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/staging/rf_pole_curb_stairs in yolov8:: 100%|██████████| 26384/26384 [00:09<00:00, 2707.05it/s]


  downloaded v6 in 29s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/rf_pole_curb_stairs in 482s

[rf_scooter_bollard] kyonggi-university-r7unh/protection_zone v4
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/staging/rf_scooter_bollard in yolov8:: 100%|██████████| 7702/7702 [00:02<00:00, 3486.30it/s]


... (7 outputs left out)


Extracting Dataset Version Zip to /content/staging/rf_cone in yolov8:: 100%|██████████| 7902/7902 [00:03<00:00, 2246.70it/s]


  downloaded v2 in 16s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/rf_cone in 227s

[rf_manhole] workspace-an3qw/manhole-j9klz v2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/staging/rf_manhole in yolov8:: 100%|██████████| 1379/1379 [00:00<00:00, 2931.37it/s]


  downloaded v2 in 3s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/rf_manhole in 32s

Roboflow section finished


## 3. Open Images V7

Loaded one class at a time so each class gets up to `OI_MAX_PER_CLASS` images, merged without duplicates, then exported with only `OI_CLASSES` boxes. The first run downloads large annotation files (about 2 GB for train).

In [4]:
# 3. Open Images V7 (detections) -> YOLO
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

fo.config.dataset_zoo_dir = "/content/fiftyone"  # zoo downloads stay on the Colab disk


def detections_field(collection) -> str:
    fields = collection.get_field_schema(embedded_doc_type=fo.Detections)
    if not fields:
        raise RuntimeError(f"{collection.name} has no detections field")
    return next(iter(fields))


def export_yolo(collection, classes: list[str], export_dir: Path, split: str) -> int:
    """Export only `classes` boxes (images without any are dropped). Returns image count."""
    field = detections_field(collection)
    view = collection.filter_labels(field, F("label").is_in(classes))
    view.export(
        export_dir=str(export_dir),
        dataset_type=fo.types.YOLOv5Dataset,
        label_field=field,
        split=split,
        classes=classes,
    )
    return len(view)


name = "openimages"
dest = RAW / name
if wanted(name) and not skip_if_done(dest):
    print(f"\n[{name}] Open Images V7, {len(OI_CLASSES)} classes, up to {OI_MAX_PER_CLASS} per class")
    local = fresh_staging(name)
    t0 = time.time()
    try:
        counts = {}
        for split, yolo_split in [("train", "train"), ("validation", "val")]:
            merged = fo.Dataset(f"oi7-{split}-merged", overwrite=True)
            for cls in OI_CLASSES:
                print(f"  {split}: loading up to {OI_MAX_PER_CLASS} images with '{cls}'")
                part = foz.load_zoo_dataset(
                    "open-images-v7",
                    split=split,
                    label_types=["detections"],
                    classes=[cls],
                    max_samples=OI_MAX_PER_CLASS,
                    shuffle=True,
                    seed=51,
                    dataset_name=f"oi7-{split}-{cls}",
                    drop_existing_dataset=True,
                )
                # One image can contain several classes: keep it once.
                merged.merge_samples(part, key_field="filepath", skip_existing=True)
            counts[yolo_split] = export_yolo(merged, OI_CLASSES, local, yolo_split)
            print(f"  {split}: exported {counts[yolo_split]} images")
        print(f"  finished in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "open-images-v7", "classes": OI_CLASSES,
            "max_per_class": OI_MAX_PER_CLASS, "images": counts, "format": "yolov5",
        })
    except Exception as e:
        record_failure(name, e)

print("\nOpen Images section finished")


invalid escape sequence '\Z'



[openimages] Open Images V7, 8 classes, up to 1500 per class
  train: loading up to 1500 images with 'Stairs'


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/content/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv' to '/content/fiftyone/open-images-v7/train/metadata/image_ids.csv'


 100% |██████|    4.8Gb/4.8Gb [6.0s elapsed, 0s remaining, 993.4Mb/s]       


INFO:eta.core.utils: 100% |██████|    4.8Gb/4.8Gb [6.0s elapsed, 0s remaining, 993.4Mb/s]       


... (266 outputs left out)

Directory '/content/staging/openimages' already exists; export will be merged with existing files


 100% |███████████████| 4481/4481 [22.7s elapsed, 0s remaining, 449.2 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 4481/4481 [22.7s elapsed, 0s remaining, 449.2 samples/s]      


  validation: exported 4481 images
  finished in 3418s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/openimages in 673s

Open Images section finished


## 4. COCO-2017 subset

In [5]:
# 4. COCO-2017 subset -> YOLO
name = "coco_subset"
dest = RAW / name
if wanted(name) and not skip_if_done(dest):
    print(f"\n[{name}] COCO-2017, {len(COCO_CLASSES)} classes, {COCO_MAX_SAMPLES}")
    local = fresh_staging(name)
    t0 = time.time()
    try:
        counts = {}
        for split, yolo_split in [("train", "train"), ("validation", "val")]:
            max_samples = COCO_MAX_SAMPLES.get(split, 0)
            if max_samples <= 0:
                continue
            print(f"  {split}: loading up to {max_samples} images")
            subset = foz.load_zoo_dataset(
                "coco-2017",
                split=split,
                label_types=["detections"],
                classes=COCO_CLASSES,
                max_samples=max_samples,
                shuffle=True,
                seed=51,
                dataset_name=f"coco-{split}-subset",
                drop_existing_dataset=True,
            )
            counts[yolo_split] = export_yolo(subset, COCO_CLASSES, local, yolo_split)
            print(f"  {split}: exported {counts[yolo_split]} images")
        print(f"  finished in {time.time() - t0:.0f}s")
        publish(local, dest, {
            "source": "coco-2017", "classes": COCO_CLASSES,
            "max_samples": COCO_MAX_SAMPLES, "images": counts, "format": "yolov5",
        })
    except Exception as e:
        record_failure(name, e)

print("\nCOCO section finished")



[coco_subset] COCO-2017, 10 classes, {'train': 5000, 'validation': 1000}
  train: loading up to 5000 images


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/content/fiftyone/coco-2017/train' if necessary


INFO:fiftyone.utils.coco:Downloading annotations to '/content/fiftyone/coco-2017/tmp-download/annotations_trainval2017.zip'


 100% |██████|    1.9Gb/1.9Gb [2.9s elapsed, 0s remaining, 454.8Mb/s]      


INFO:eta.core.utils: 100% |██████|    1.9Gb/1.9Gb [2.9s elapsed, 0s remaining, 454.8Mb/s]      


Extracting annotations to '/content/fiftyone/coco-2017/raw/instances_train2017.json'


... (33 outputs left out)

Directory '/content/staging/coco_subset' already exists; export will be merged with existing files


 100% |███████████████| 1000/1000 [4.1s elapsed, 0s remaining, 276.3 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 1000/1000 [4.1s elapsed, 0s remaining, 276.3 samples/s]      


  validation: exported 1000 images
  finished in 876s
  copied to /content/drive/MyDrive/clear_vision/datasets/raw/coco_subset in 211s

COCO section finished


## 5. Mendeley

In [6]:
# 5. Mendeley: unzip uploaded archives in place
mendeley = RAW / "mendeley"
zips = sorted(mendeley.glob("*.zip")) if mendeley.exists() and wanted("mendeley") else []

if not wanted("mendeley"):
    print("Mendeley not in ONLY, skipping")
elif not zips:
    print(f"No .zip files in {mendeley}")
    print("Download the Mendeley datasets in a browser, upload the .zip files there, re-run this cell.")

for zip_path in zips:
    target = zip_path.with_suffix("")  # raw/mendeley/<zip name>/
    if skip_if_done(target):
        continue
    print(f"\n[mendeley/{target.name}] unzipping {zip_path.name}")
    t0 = time.time()
    try:
        shutil.rmtree(target, ignore_errors=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)  # extractall refuses paths that escape `target`
        (target / DONE).write_text(json.dumps({"source": "mendeley", "zip": zip_path.name}, indent=2))
        print(f"  unzipped in {time.time() - t0:.0f}s")
    except Exception as e:
        record_failure(f"mendeley/{target.name}", e)

print("\nMendeley section finished")



[mendeley/Obstacles Avoidance Assistance for Visually Impair] unzipping Obstacles Avoidance Assistance for Visually Impair.zip
  unzipped in 39s

Mendeley section finished


## 6. Summary

In [7]:
# 6. Summary table
import pandas as pd
import yaml
from IPython.display import display

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def count_files(root: Path) -> tuple[int, int]:
    """(images, YOLO label files). Label files are .txt files inside a `labels` folder."""
    images = labels = 0
    for dirpath, _, filenames in os.walk(root):
        in_labels = "labels" in Path(dirpath).relative_to(root).parts
        for filename in filenames:
            ext = Path(filename).suffix.lower()
            if ext in IMAGE_EXTS:
                images += 1
            elif ext == ".txt" and in_labels:
                labels += 1
    return images, labels


def class_names(root: Path) -> tuple[list[str], str]:
    """Class names from data.yaml / dataset.yaml, else from a classes file."""
    for pattern in ("data.yaml", "dataset.yaml"):
        for path in sorted(root.rglob(pattern)):
            try:
                names = (yaml.safe_load(path.read_text()) or {}).get("names")
            except yaml.YAMLError:
                continue
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            if names:
                return [str(n) for n in names], path.name
    for pattern in ("classes.txt", "_classes.txt", "*.names"):
        for path in sorted(root.rglob(pattern)):
            names = [line.strip() for line in path.read_text().splitlines() if line.strip()]
            if names:
                return names, path.name
    return [], "-"


def dataset_dirs() -> list[Path]:
    dirs = []
    for path in sorted(p for p in RAW.iterdir() if p.is_dir() and not p.name.startswith(".")):
        if path.name == "mendeley":
            dirs += sorted(p for p in path.iterdir() if p.is_dir() and not p.name.startswith("."))
        else:
            dirs.append(path)
    return dirs


rows = []
for root in dataset_dirs():
    name = str(root.relative_to(RAW))
    images, labels = count_files(root)
    names, source = class_names(root)
    rows.append({
        "dataset": name,
        "status": "done" if is_done(root) else "PARTIAL",
        "images": images,
        "label files": labels,
        "classes": len(names),
        "class names": ", ".join(names) if names else "(none found)",
        "classes from": source,
    })
for name, error in FAILURES.items():
    if not any(row["dataset"] == name for row in rows):
        rows.append({"dataset": name, "status": "FAILED", "images": 0, "label files": 0,
                     "classes": 0, "class names": error, "classes from": "-"})

pd.set_option("display.max_colwidth", None)
summary = pd.DataFrame(rows)
display(summary)
print(f"\n{len(rows)} datasets, {summary['images'].sum() if rows else 0} images in {RAW}")

if FAILURES:
    print(f"\n{len(FAILURES)} FAILED (re-run all cells to retry only these):")
    for name, error in FAILURES.items():
        print(f"  - {name}: {error}")
else:
    print("No failures.")


,dataset,status,images,label files,classes,class names,classes from
0,coco_subset,done,6000,6000,10,"person, bicycle, car, motorcycle, bus, truck, dog, bench, fire hydrant, traffic light",dataset.yaml
1,mendeley/Obstacles Avoidance Assistance for Visually Impair,done,1250,1250,4,"bump, fence, hole, pole",data.yaml
2,openimages,done,14337,14337,8,"Stairs, Traffic light, Fire hydrant, Bench, Bicycle, Person, Car, Dog",dataset.yaml
3,rf_bollard,done,67,67,1,bollard,data.yaml
4,rf_cone,done,3945,3945,1,cone,data.yaml
5,rf_crosswalk,done,12524,12524,4,"2, crosswalk, pedestrian, pedestrian-crossing",data.yaml
6,rf_manhole,done,685,685,1,manhole cover,data.yaml
7,rf_pole_curb_stairs,done,13186,13186,3,"curb, pole, stairs",data.yaml
8,rf_pothole,done,2742,2742,1,pothole,data.yaml
9,rf_scooter_bollard,done,3845,3845,5,"Bicycle, Electric-Scooter, bollard, motorcycle, person",data.yaml



12 datasets, 60148 images in /content/drive/MyDrive/clear_vision/datasets/raw

1 FAILED (re-run all cells to retry only these):
  - rf_sidewalk_obstacles: RuntimeError: blindspot-fmcu0/sidewalk-obstacles has no versions
